In [0]:
# ---- Silver config  ----
CATALOG = "dbr_dev"
SCHEMA  = "live_transit_monitor"

BRONZE   = f"{CATALOG}.{SCHEMA}.gps_data"
VEHICLES = f"{CATALOG}.{SCHEMA}.bronze_vehicles"
ROUTES   = f"{CATALOG}.{SCHEMA}.bronze_gtfs_routes"
SILVER   = f"{CATALOG}.{SCHEMA}.gps_positions_silver"

In [0]:
bronze = spark.read.table(BRONZE)
display(df.limit(10))   
bronze.selectExpr("min(event_time) min_t", "max(event_time) max_t",
              "count(distinct vehicleId) vehicles").show()

In [0]:
bronze.printSchema()

# lastUpdate needs to be changed to timestamp

In [0]:
# Checking if there are any nulls in the table
from pyspark.sql.functions import col, count, when

bronze.select([count(when(col(c).isNull(), c)).alias(c) for c in df.columns]).show()

# Nulls when a vehicle is not on a trip (returning to car barn)


In [0]:
from pyspark.sql import functions as F, Window

bronze = spark.read.table(BRONZE_GPS_DATA)

silver = (bronze
    .dropDuplicates(["vehicleId", "generated"])
    .withColumn("scheduled_start", F.to_timestamp("scheduledTripStartTime"))
    .withColumn("delay_min", F.round(F.col("delay") / 60.0, 1))
    .withColumn("has_trip", F.col("tripId").isNotNull())   # vehicle currently on a scheduled trip
    .withColumn("delay_bucket",
        F.when(F.col("delay") < -60, "early")
         .when(F.col("delay") <= 120, "on_time")
         .otherwise("delayed"))
    .withColumn("is_moving", F.col("speed") > 0)
    .withColumn("gps_ok",    F.col("gpsQuality") > 0)
    .filter(F.col("lat").isNotNull() & F.col("lon").isNotNull())
)

# ===== TODO: ENRICHMENT — wpiąć, gdy Radek dostarczy batch, najpierw sprawdzić jego tabele pod kątem nulli, duplikatów i schema =====
# silver = (silver
#     .join(gtfs_trips, on="tripId",   how="left")   # route long name, stop, scheduled times
#     .join(vehicles,   on="vehicleId", how="left")) # vehicle type (tram/bus), capacity
# ===============================================================

(silver.write.format("delta").mode("overwrite").option("overwriteSchema", "true")
    .saveAsTable(f"{CATALOG}.{SCHEMA}.gps_positions_silver"))